In [8]:
from pathlib import Path
import pandas as pd

In [9]:
folder = Path("/home/ibroto/Documents/UPF_SMC/MIR/flamenco-mir/SourceSeparation/demucs_output")
wav_files = list(folder.glob("**/voice.wav"))
print(f"{len(wav_files)} wav files found.")

5 wav files found.


In [10]:
import torchaudio
import pesto
import torch
import soundfile as sf

wav, sr = sf.read(wav_files[0])
wav = torch.tensor(wav).float()

print(sr)
# convert to channels x samples
if wav.ndim == 1:
    wav = wav.unsqueeze(0)
else:
    wav = wav.T

# Load audio (ensure mono; stereo channels are treated as separate batch dimensions)
#x, sr = torchaudio.load(wa)
wav = wav.mean(dim=0)  # PESTO takes mono audio as input

# Predict pitch. x can be (num_samples) or (batch, num_samples)
timesteps, pitch, confidence, activations = pesto.predict(wav, sr, step_size=2)

# Predicting from multiple files:
#pesto.predict_from_files(["example1.wav", "example2.mp3"], export_format=["csv"])

ImportError: numpy.core.multiarray failed to import (auto-generated because you didn't call 'numpy.import_array()' after cimporting numpy; use '<void>numpy._import_array' to disable if you are certain you don't need it).

In [ ]:
print(len(timesteps))
import numpy as np
step_size=1
pitch = np.where(confidence < 0.5, -220, pitch)

f0_pred = pd.DataFrame(data={
    'timestamp':timesteps* (step_size / 1000.0),
    'freq': pitch,
    'confidence':confidence
})
f0_pred

33294


,timestamp,freq,confidence
0,0.000000,-220.0,0.090661
1,0.025000,-220.0,0.077913
2,0.050000,-220.0,0.070254
3,0.075000,-220.0,0.066402
4,0.100000,-220.0,0.051932
...,...,...,...
33289,832.224976,-220.0,0.070539
33290,832.250000,-220.0,0.067382
33291,832.274963,-220.0,0.068469
33292,832.299988,-220.0,0.070823


In [23]:
import pandas as pd
folder = Path("/home/ibroto/Documents/UPF_SMC/MIR/flamenco-mir/data/cante2midi_f0/")
gt_f0_files = list(folder.glob("*.csv"))
f0_ref = pd.read_csv(gt_f0_files[6], names=['timestamp', 'freq'])
f0_ref

,timestamp,freq
0,0.023220,-220.0
1,0.026122,-220.0
2,0.029025,-220.0
3,0.031927,-220.0
4,0.034830,-220.0
...,...,...
57581,167.151746,-220.0
57582,167.154649,-220.0
57583,167.157551,-220.0
57584,167.160454,-220.0


In [24]:
wav_files[0]

PosixPath('/home/ibroto/Documents/UPF_SMC/MIR/flamenco-mir/SourceSeparation/demucs_output/08_PericonDeCadiz_MeMetieronEnUnVapor/voice.wav')

In [25]:
import numpy as np
from scipy.interpolate import interp1d

def compute_f0_metrics(timesteps_pred, pitch_pred, confidence_pred, 
                       timestamps_gt, freq_gt, 
                       voicing_threshold=0.5, cent_tolerance=50):
    """
    Compute F0 estimation metrics.
    
    Args:
        timesteps_pred: predicted timestamps (from PESTO)
        pitch_pred: predicted pitch in Hz (from PESTO)
        confidence_pred: prediction confidence (from PESTO)
        timestamps_gt: ground truth timestamps
        freq_gt: ground truth frequencies (0 or NaN for unvoiced)
        voicing_threshold: confidence threshold for voiced/unvoiced decision
        cent_tolerance: tolerance in cents for RPA/RCA (typically 50)
    
    Returns:
        dict with accuracy metrics
    """
    
    # Convert to numpy arrays
    timesteps_pred = np.array(timesteps_pred)
    pitch_pred = np.array(pitch_pred)
    confidence_pred = np.array(confidence_pred)
    timestamps_gt = np.array(timestamps_gt)
    freq_gt = np.array(freq_gt)
    
    # Interpolate ground truth to match prediction timestamps
    # Only interpolate where ground truth is voiced (freq > 0)
    voiced_gt_mask = (freq_gt > 0) & ~np.isnan(freq_gt)
    
    if np.sum(voiced_gt_mask) < 2:
        return {"error": "Not enough voiced frames in ground truth"}
    
    # Interpolate ground truth frequencies
    interp_func = interp1d(timestamps_gt[voiced_gt_mask], 
                           freq_gt[voiced_gt_mask],
                           kind='linear', 
                           bounds_error=False,
                           fill_value=0)
    
    freq_gt_aligned = interp_func(timesteps_pred)
    
    # Determine voicing for predictions and ground truth
    voiced_pred = confidence_pred > voicing_threshold
    voiced_gt_aligned = freq_gt_aligned > 0
    
    # Only evaluate frames where both have voicing decisions
    valid_frames = ~np.isnan(freq_gt_aligned)
    
    # Raw Pitch Accuracy (RPA)
    # Compute cent errors only for voiced frames in both
    both_voiced = voiced_pred & voiced_gt_aligned & valid_frames
    
    if np.sum(both_voiced) > 0:
        # Convert to cents: 1200 * log2(f_est / f_ref)
        cent_errors = 1200 * np.log2(pitch_pred[both_voiced] / freq_gt_aligned[both_voiced])
        rpa = np.mean(np.abs(cent_errors) < cent_tolerance) * 100
        
        # Raw Chroma Accuracy (RCA) - ignore octave errors
        # Map errors to [-600, 600] cents range
        chroma_errors = np.mod(cent_errors + 600, 1200) - 600
        rca = np.mean(np.abs(chroma_errors) < cent_tolerance) * 100
        
        mean_abs_error_cents = np.mean(np.abs(cent_errors))
        median_abs_error_cents = np.median(np.abs(cent_errors))
    else:
        rpa = rca = mean_abs_error_cents = median_abs_error_cents = np.nan
    
    # Voicing metrics
    voiced_recall = np.sum(voiced_pred & voiced_gt_aligned) / np.sum(voiced_gt_aligned) * 100 if np.sum(voiced_gt_aligned) > 0 else np.nan
    voiced_precision = np.sum(voiced_pred & voiced_gt_aligned) / np.sum(voiced_pred) * 100 if np.sum(voiced_pred) > 0 else np.nan
    
    return {
        "RPA": rpa,  # Raw Pitch Accuracy
        "RCA": rca,  # Raw Chroma Accuracy
        "mean_abs_error_cents": mean_abs_error_cents,
        "median_abs_error_cents": median_abs_error_cents,
        "voiced_recall": voiced_recall,
        "voiced_precision": voiced_precision,
        "num_voiced_frames_gt": np.sum(voiced_gt_aligned),
        "num_voiced_frames_pred": np.sum(voiced_pred),
        "num_both_voiced": np.sum(both_voiced)
    }

# Example usage:
metrics = compute_f0_metrics(
    timesteps, pitch, confidence,
    f0_ref['timestamp'].values, f0_ref['freq'].values
)

print("F0 Estimation Metrics:")
for key, value in metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.2f}")
    else:
        print(f"  {key}: {value}")

F0 Estimation Metrics:
  RPA: nan
  RCA: nan
  mean_abs_error_cents: nan
  median_abs_error_cents: nan
  voiced_recall: 0.00
  voiced_precision: 0.00
  num_voiced_frames_gt: 16
  num_voiced_frames_pred: 8452
  num_both_voiced: 0
